# Demo 4 — Confrontare le feature sulla validazione

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

**Risultati registrati:** questa demo esplora il manifest e le predizioni dell’esperimento già eseguito. Non addestra la CNN. Le metriche e i grafici vengono ricalcolati nel browser; i clip degli errori sono scaricati solo su richiesta.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
metrica = "macro_f1"              # "accuracy" oppure "macro_f1"
risultati = al.load_classification_results(DEMO_ASSETS / "classification-results.json")
assert metrica in ("accuracy","macro_f1")
veri = [r["label"] for r in risultati["splits"]["val"]]
nomi = ["Dummy","RF media","RF media+std"]
valori = []
for nome in nomi:
    pred = risultati["models"][nome]["val_predictions"]
    misure = al.classification_metrics(veri,pred,n_classes=len(risultati["classes"]))
    valori.append(misure[metrica])
    print(f"{nome:14s} accuratezza {misure['accuracy']:.3f}; macro-F1 {misure['macro_f1']:.3f}")
fig = go.Figure(go.Bar(x=nomi,y=valori))
fig.update_layout(title=f"Validazione — {metrica}, predizioni registrate",yaxis_title=metrica,yaxis_range=[0,1],height=360)
fig.show()
print("Le metriche sono ricalcolate dalle predizioni salvate, sugli stessi clip di validazione.")
print("Nel browser non viene addestrato o selezionato alcun modello.")
